# 가우시안 프로세스 기반 공력 보정 연구
## GP-based Aerodynamic Correction for Missile Flight

---

> **개인 연구 노트** | Personal Research Note
>
> 이 노트북은 GP regression을 공력 모델 보정에 적용해본 개인 연구입니다.  
> *This notebook documents a personal study applying Gaussian Process regression to aerodynamic model correction.*

---

**연구 동기 (Motivation)**

공력 계수는 풍동시험(Wind Tunnel Test)이나 CFD로 얻지만, 현실에는 항상 **불확실성**이 존재합니다.  
비행시험 데이터로 모델을 보정할 수 있을까? — 그 가능성을 탐색해봤습니다.

*Aerodynamic coefficients come from wind tunnel tests or CFD, but uncertainty always exists in practice.  
Can we correct the model using flight test data? — This notebook explores that possibility.*

**핵심 질문:** 머신러닝(GP)을 유도탄 공력 모델에 적용할 수 있을까?  
*Key question: Can machine learning (GP) be applied to missile aerodynamic models?*

## 1. 문제 정의 (Problem Definition)

### 공력 모델의 한계

**일반적인 선형 공력 모델:**
$$C_D(\\alpha, M) = C_{D_0}(M) + C_{D_\\alpha}(M) \\cdot \\alpha^2$$
$$C_N(\\alpha, M) = C_{N_\\alpha}(M) \\cdot \\alpha$$

이 모델은 **소받음각(small angle of attack)** 및 **아음속/초음속** 영역에서는 잘 맞지만:

| 영역 | 문제점 |
|------|--------|
| 고받음각 (α > 15°) | 비선형성 증가, 와류(vortex) 효과 |
| 천음속 (0.8 < M < 1.2) | 충격파 간섭, 급격한 변화 |
| 기동 중 동적 효과 | 공력 미분계수의 비정상성 |

*The linear model works well for small angles and sub/supersonic flight, but breaks down at high AoA, transonic regime, and during aggressive maneuvers.*

### 아이디어

$$C_D^{\\text{true}}(\\alpha, M) = C_D^{\\text{model}}(\\alpha, M) + \\underbrace{\\Delta C_D(\\alpha, M)}_{\\text{GP로 학습}}$$

비행시험에서 얻은 **잔차(residual)**를 GP로 학습 → 보정된 모델 구성  
*Learn the residual error from flight test data using GP, then apply as a correction.*

## 2. Gaussian Process Regression 기초

### 핵심 개념

GP는 **함수 위의 분포(distribution over functions)**입니다.  
*A GP is a distribution over functions — instead of fitting a single function, we maintain a distribution.*

$$f(\\mathbf{x}) \\sim \\mathcal{GP}(m(\\mathbf{x}),\\, k(\\mathbf{x}, \\mathbf{x}'))$$

여기서:
- $m(\\mathbf{x})$: mean function (보통 0으로 설정)
- $k(\\mathbf{x}, \\mathbf{x}')$: kernel (covariance) function — **데이터 간의 유사도**를 정의

---

### Kernel Functions

**RBF (Radial Basis Function) / Squared Exponential Kernel:**
$$k_{\\text{RBF}}(\\mathbf{x}, \\mathbf{x}') = \\sigma_f^2 \\exp\\!\\left(-\\frac{\\|\\mathbf{x} - \\mathbf{x}'\\|^2}{2l^2}\\right)$$

- $\\sigma_f^2$: signal variance (출력 스케일)
- $l$: length scale (입력 공간에서의 상관 거리)

**Matérn 5/2 Kernel** (공학 문제에 자주 쓰임):
$$k_{\\text{Matérn}}(r) = \\sigma_f^2\\left(1 + \\frac{\\sqrt{5}\\,r}{l} + \\frac{5r^2}{3l^2}\\right)\\exp\\!\\left(-\\frac{\\sqrt{5}\\,r}{l}\\right)$$

---

### 사후 분포 (Posterior Distribution)

학습 데이터 $\\mathcal{D} = \\{\\mathbf{X}, \\mathbf{y}\\}$ 와 노이즈 $\\epsilon \\sim \\mathcal{N}(0, \\sigma_n^2)$ 가 있을 때,  
테스트 입력 $\\mathbf{X}_*$ 에서의 예측 분포:

$$f_* \\mid \\mathbf{X}, \\mathbf{y}, \\mathbf{X}_* \\sim \\mathcal{N}(\\boldsymbol{\\mu}_*, \\boldsymbol{\\Sigma}_*)$$

$$\\boldsymbol{\\mu}_* = K(\\mathbf{X}_*, \\mathbf{X})\\,\\bigl[K(\\mathbf{X}, \\mathbf{X}) + \\sigma_n^2 I\\bigr]^{-1}\\mathbf{y}$$

$$\\boldsymbol{\\Sigma}_* = K(\\mathbf{X}_*, \\mathbf{X}_*) - K(\\mathbf{X}_*, \\mathbf{X})\\,\\bigl[K(\\mathbf{X}, \\mathbf{X}) + \\sigma_n^2 I\\bigr]^{-1}K(\\mathbf{X}, \\mathbf{X}_*)$$

**GP의 핵심 장점:** 예측값뿐만 아니라 **불확실성(uncertainty)**도 함께 제공!  
*Key advantage: GP gives not just a prediction, but also a calibrated uncertainty estimate.*

---

### Log Marginal Likelihood (하이퍼파라미터 최적화)

$$\\log p(\\mathbf{y} \\mid \\mathbf{X}, \\boldsymbol{\\theta}) = -\\frac{1}{2}\\mathbf{y}^\\top K_y^{-1}\\mathbf{y} - \\frac{1}{2}\\log|K_y| - \\frac{n}{2}\\log 2\\pi$$

여기서 $K_y = K(\\mathbf{X}, \\mathbf{X}) + \\sigma_n^2 I$

In [ ]:
# ============================================================
# GP from scratch — scikit-learn 없이 직접 구현해서 원리 이해
# Implementing GP from scratch using only numpy/scipy
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# ── Kernel Functions ─────────────────────────────────────────

def rbf_kernel(X1, X2, length_scale=1.0, signal_var=1.0):
    """RBF (Squared Exponential) kernel.
    k(x, x') = sigma_f^2 * exp(-||x-x'||^2 / (2*l^2))
    """
    X1 = np.atleast_2d(X1)
    X2 = np.atleast_2d(X2)
    diff = X1[:, np.newaxis, :] - X2[np.newaxis, :, :]  # (n1, n2, d)
    sq_dist = np.sum(diff**2, axis=-1)                   # (n1, n2)
    return signal_var * np.exp(-0.5 * sq_dist / length_scale**2)


def matern52_kernel(X1, X2, length_scale=1.0, signal_var=1.0):
    """Matern 5/2 kernel — less smooth than RBF, often better for physical systems."""
    X1 = np.atleast_2d(X1)
    X2 = np.atleast_2d(X2)
    diff = X1[:, np.newaxis, :] - X2[np.newaxis, :, :]
    r = np.sqrt(np.sum(diff**2, axis=-1) + 1e-12)
    sqrt5_r_l = np.sqrt(5) * r / length_scale
    return signal_var * (1 + sqrt5_r_l + 5*r**2/(3*length_scale**2)) * np.exp(-sqrt5_r_l)


# ── GP Class ─────────────────────────────────────────────────

class GaussianProcess:
    """Gaussian Process Regressor — numpy only, no scikit-learn."""

    def __init__(self, kernel='rbf', length_scale=1.0, signal_var=1.0, noise_var=1e-3):
        self.kernel_name  = kernel
        self.length_scale = length_scale
        self.signal_var   = signal_var
        self.noise_var    = noise_var
        self.X_train = None
        self.y_train = None
        self.K_inv   = None
        self.alpha   = None

    def _kernel(self, X1, X2):
        if self.kernel_name == 'rbf':
            return rbf_kernel(X1, X2, self.length_scale, self.signal_var)
        elif self.kernel_name == 'matern52':
            return matern52_kernel(X1, X2, self.length_scale, self.signal_var)
        else:
            raise ValueError(f'Unknown kernel: {self.kernel_name}')

    def fit(self, X_train, y_train):
        """Compute and cache (K + sigma_n^2 * I)^{-1} for fast prediction."""
        self.X_train = np.atleast_2d(X_train)
        self.y_train = np.asarray(y_train).ravel()
        n = len(self.y_train)
        K = self._kernel(self.X_train, self.X_train)
        K_noisy = K + self.noise_var * np.eye(n)
        try:
            L = np.linalg.cholesky(K_noisy)
            self.K_inv = np.linalg.solve(L.T, np.linalg.solve(L, np.eye(n)))
        except np.linalg.LinAlgError:
            self.K_inv = np.linalg.inv(K_noisy + 1e-6 * np.eye(n))
        self.alpha = self.K_inv @ self.y_train
        return self

    def predict(self, X_test, return_std=True):
        """GP posterior prediction. Returns: mean, std (if return_std=True)"""
        X_test  = np.atleast_2d(X_test)
        K_star  = self._kernel(X_test, self.X_train)   # (n_test, n_train)
        K_ss    = self._kernel(X_test, X_test)          # (n_test, n_test)
        mu_star = K_star @ self.alpha
        if return_std:
            cov_star = K_ss - K_star @ self.K_inv @ K_star.T
            var_star = np.maximum(np.diag(cov_star), 0.0)
            return mu_star, np.sqrt(var_star)
        return mu_star

    def log_marginal_likelihood(self, params=None):
        """Compute log p(y | X, theta) for hyperparameter optimization."""
        if params is not None:
            self.length_scale = np.exp(params[0])
            self.signal_var   = np.exp(params[1])
            self.noise_var    = np.exp(params[2])
            self.fit(self.X_train, self.y_train)
        n = len(self.y_train)
        K = self._kernel(self.X_train, self.X_train)
        K_noisy = K + self.noise_var * np.eye(n)
        try:
            L = np.linalg.cholesky(K_noisy)
            log_det = 2 * np.sum(np.log(np.diag(L)))
            alpha_v = np.linalg.solve(L.T, np.linalg.solve(L, self.y_train))
        except np.linalg.LinAlgError:
            return -1e10
        return (-0.5 * self.y_train @ alpha_v
                - 0.5 * log_det
                - 0.5 * n * np.log(2 * np.pi))

    def optimize_hyperparams(self, n_restarts=5):
        """Optimize hyperparameters by maximizing log marginal likelihood."""
        best_lml, best_params = -np.inf, None
        rng = np.random.default_rng(42)

        def neg_lml(log_params):
            return -self.log_marginal_likelihood(log_params)

        for _ in range(n_restarts):
            x0 = rng.uniform(low=[-2, -2, -4], high=[2, 2, -1])
            result = minimize(neg_lml, x0, method='L-BFGS-B',
                              bounds=[(-3, 3), (-3, 3), (-8, 0)])
            if result.success and -result.fun > best_lml:
                best_lml   = -result.fun
                best_params = result.x

        if best_params is not None:
            self.length_scale = np.exp(best_params[0])
            self.signal_var   = np.exp(best_params[1])
            self.noise_var    = np.exp(best_params[2])
            self.fit(self.X_train, self.y_train)
            print(f'Optimized: length_scale={self.length_scale:.4f}, '
                  f'signal_var={self.signal_var:.4f}, noise_var={self.noise_var:.6f}')
        return self


# ── 1D 검증 예제: noisy sin(x) 피팅 ─────────────────────────
print('=' * 55)
print('1D 검증: noisy sin(x) 피팅')
print('GP from scratch — numpy only')
print('=' * 55)

rng = np.random.default_rng(0)
X_1d = rng.uniform(0, 2*np.pi, size=15).reshape(-1, 1)
y_1d = np.sin(X_1d.ravel()) + rng.normal(0, 0.15, size=15)

X_test_1d = np.linspace(-0.5, 2*np.pi + 0.5, 200).reshape(-1, 1)

gp_1d = GaussianProcess(kernel='rbf', length_scale=1.0, signal_var=1.0, noise_var=0.02)
gp_1d.fit(X_1d, y_1d)
gp_1d.optimize_hyperparams(n_restarts=8)

mu, sigma = gp_1d.predict(X_test_1d)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(X_test_1d.ravel(),
                mu - 2*sigma, mu + 2*sigma,
                alpha=0.25, color='steelblue', label='+/-2sigma 불확실성 구간')
ax.plot(X_test_1d, mu, color='steelblue', lw=2, label='GP 예측 평균')
ax.plot(X_test_1d, np.sin(X_test_1d), 'k--', lw=1.5, alpha=0.6, label='True sin(x)')
ax.scatter(X_1d, y_1d, c='tomato', s=60, zorder=5, label='학습 데이터 (노이즈 포함)')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('GP Regression: 1D 검증 — noisy sin(x)\n'
             '데이터가 없는 영역에서 불확실성이 커지는 것을 확인', pad=12)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print('\n핵심 관찰:')
print('  - 데이터가 있는 영역: 예측 정확, 불확실성 낮음')
print('  - 데이터 밖 외삽 영역: 불확실성이 급격히 증가')
print('  -> 이 특성이 공학적으로 매우 유용함 (보수적 설계 가능)')

## 3. 공력 모델 불확실성 시뮬레이션

### 시뮬레이션 설정

실제 비행 데이터 대신, **의도적으로 '진짜' 공력과 '모델' 공력의 차이를 만들어봅니다.**  
*Instead of real flight data, we synthesize the discrepancy between 'true' and 'model' aerodynamics.*

$$C_D^{\\text{true}}(\\alpha, M) = C_{D_0}(M) + C_{D_\\alpha} \\cdot \\alpha^2 + \\underbrace{C_{D_{\\alpha^4}} \\cdot \\alpha^4}_{\\text{고차 항}} + \\underbrace{\\Delta C_D^{\\text{transonic}}(M)}_{\\text{천음속 결합}}$$

$$C_D^{\\text{model}}(\\alpha, M) = C_{D_0}(M) + C_{D_\\alpha} \\cdot \\alpha^2 \\quad \\text{(선형 근사)}$$

**모델 오차** = True - Model → GP로 학습할 타깃

In [ ]:
# ============================================================
# 공력 모델 생성 — True (nonlinear) vs Model (linear approx)
# ============================================================

def cd_base(mach):
    """Base drag as function of Mach — transonic bump near M=1."""
    cd0 = 0.015 + 0.005 * mach**2
    transonic_bump = 0.035 * np.exp(-((mach - 1.0) / 0.15)**2)
    return cd0 + transonic_bump


def cd_true(alpha_deg, mach):
    """True CD: nonlinear — includes high-order AoA and transonic-AoA coupling."""
    alpha_rad = np.deg2rad(alpha_deg)
    cd0   = cd_base(mach)
    cd_q  = 0.18 * alpha_rad**2
    cd_4  = 0.45 * alpha_rad**4
    cd_tc = 0.12 * alpha_rad**2 * np.exp(-((mach - 1.0) / 0.2)**2)
    return cd0 + cd_q + cd_4 + cd_tc


def cd_model(alpha_deg, mach):
    """Linear aero model — quadratic AoA only, no transonic coupling."""
    alpha_rad = np.deg2rad(alpha_deg)
    return cd_base(mach) + 0.18 * alpha_rad**2


# Grid for visualization
alpha_grid = np.linspace(0, 25, 60)
mach_grid  = np.linspace(0.5, 2.5, 60)
A, M = np.meshgrid(alpha_grid, mach_grid)

CD_true_grid  = cd_true(A, M)
CD_model_grid = cd_model(A, M)
CD_error      = CD_true_grid - CD_model_grid

# ── Plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
levels_cd  = np.linspace(0, 0.35, 30)
levels_err = np.linspace(-0.01, 0.12, 30)

c0 = axes[0].contourf(A, M, CD_true_grid,  levels=levels_cd, cmap='plasma')
axes[0].set_title('True CD(alpha, M)\n(비선형 + 천음속 결합항)', pad=10)
axes[0].set_xlabel('받음각 alpha (deg)')
axes[0].set_ylabel('마하수 M')
plt.colorbar(c0, ax=axes[0], label='CD')

c1 = axes[1].contourf(A, M, CD_model_grid, levels=levels_cd, cmap='plasma')
axes[1].set_title('Linear Model CD(alpha, M)\n(이차항까지만)', pad=10)
axes[1].set_xlabel('받음각 alpha (deg)')
axes[1].set_ylabel('마하수 M')
plt.colorbar(c1, ax=axes[1], label='CD')

c2 = axes[2].contourf(A, M, CD_error, levels=levels_err, cmap='RdYlBu_r')
axes[2].set_title('오차 Delta_CD = True - Model\n(고받음각+천음속에서 최대)', pad=10)
axes[2].set_xlabel('받음각 alpha (deg)')
axes[2].set_ylabel('마하수 M')
plt.colorbar(c2, ax=axes[2], label='Delta_CD')

plt.suptitle('공력 계수 비교: True vs Linear Model', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

max_idx = CD_error.ravel().argmax()
print(f'최대 오차: {CD_error.max():.4f}  '
      f'(alpha={A.ravel()[max_idx]:.1f} deg, M={M.ravel()[max_idx]:.2f})')
print(f'RMSE (전 영역): {np.sqrt(np.mean(CD_error**2)):.4f}')
print()
print('관찰: 선형 모델은 고받음각(alpha>15deg) + 천음속(M~1) 영역에서 오차 집중')

# 학습 데이터 샘플링 (비행시험 포인트 시뮬레이션)
alpha_scale = 25.0
mach_scale  = 2.5
rng_data = np.random.default_rng(7)
n_train  = 30
noise_std = 0.003
alpha_tr = rng_data.uniform(2, 22, n_train)
mach_tr  = rng_data.uniform(0.6, 2.2, n_train)
delta_cd_tr = (cd_true(alpha_tr, mach_tr)
               - cd_model(alpha_tr, mach_tr)
               + rng_data.normal(0, noise_std, n_train))

print(f'\n학습 데이터 수: {n_train}개 (측정 노이즈 sigma={noise_std})')
print('(실제 비행시험에서는 이보다 훨씬 적을 수 있음 — GP의 데이터 효율성이 중요)')

In [ ]:
# ============================================================
# GP 학습 — 공력 오차 Delta_CD(alpha, M) 학습
# ============================================================

# 입력 정규화 (중요: 두 입력의 스케일이 다름)
X_train_gp = np.column_stack([
    alpha_tr / alpha_scale,
    mach_tr  / mach_scale
])
y_train_gp = delta_cd_tr

print('GP 학습 중...')
print('(하이퍼파라미터 최적화: log marginal likelihood 최대화)')

gp_aero = GaussianProcess(kernel='matern52',
                           length_scale=0.3,
                           signal_var=0.01,
                           noise_var=noise_std**2)
gp_aero.fit(X_train_gp, y_train_gp)
gp_aero.optimize_hyperparams(n_restarts=10)

# 예측: 전체 (alpha, M) 그리드
X_grid_gp = np.column_stack([
    A.ravel() / alpha_scale,
    M.ravel() / mach_scale
])
mu_grid, sigma_grid = gp_aero.predict(X_grid_gp)
mu_grid    = mu_grid.reshape(A.shape)
sigma_grid = sigma_grid.reshape(A.shape)

# ── Plot: True error vs GP prediction ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

c0 = axes[0].contourf(A, M, CD_error, levels=levels_err, cmap='RdYlBu_r')
axes[0].scatter(alpha_tr, mach_tr, c='white', s=40,
                edgecolors='k', lw=0.8, zorder=5, label='학습 데이터')
axes[0].set_title('True 오차 Delta_CD', pad=10)
axes[0].set_xlabel('받음각 alpha (deg)')
axes[0].set_ylabel('마하수 M')
axes[0].legend(fontsize=9)
plt.colorbar(c0, ax=axes[0])

c1 = axes[1].contourf(A, M, mu_grid, levels=levels_err, cmap='RdYlBu_r')
axes[1].scatter(alpha_tr, mach_tr, c='white', s=40,
                edgecolors='k', lw=0.8, zorder=5)
axes[1].set_title('GP 예측 Delta_CD_hat', pad=10)
axes[1].set_xlabel('받음각 alpha (deg)')
axes[1].set_ylabel('마하수 M')
plt.colorbar(c1, ax=axes[1])

sigma_max = float(sigma_grid.max())
c2 = axes[2].contourf(A, M, sigma_grid,
                       levels=np.linspace(0, sigma_max, 25),
                       cmap='Greens')
axes[2].scatter(alpha_tr, mach_tr, c='red', s=40,
                edgecolors='k', lw=0.8, zorder=5)
axes[2].set_title('GP 예측 불확실성 sigma\n(데이터 없는 영역 = 높은 불확실성)', pad=10)
axes[2].set_xlabel('받음각 alpha (deg)')
axes[2].set_ylabel('마하수 M')
plt.colorbar(c2, ax=axes[2], label='std dev')

plt.suptitle('GP 학습 결과: Delta_CD(alpha, M) 공력 오차 보정', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

gp_residual = CD_error - mu_grid
rmse_before = np.sqrt(np.mean(CD_error**2))
rmse_after  = np.sqrt(np.mean(gp_residual**2))
print('\n--- GP 예측 성능 (전체 그리드) ---')
print(f'True 오차  RMSE (보정 전): {rmse_before:.5f}')
print(f'GP 잔차    RMSE (보정 후): {rmse_after:.5f}')
print(f'개선율: {(1 - rmse_after/rmse_before)*100:.1f}%')

## 4. 보정된 공력 모델 성능

GP 보정 후 모델이 **테스트 포인트**에서 실제 공력을 얼마나 잘 맞추는지 정량 평가합니다.

*Quantitative evaluation of GP-corrected model on held-out test points.*

$$C_D^{\\text{corrected}}(\\alpha, M) = C_D^{\\text{model}}(\\alpha, M) + \\hat{\\Delta C}_D^{\\text{GP}}(\\alpha, M)$$

In [ ]:
# ============================================================
# 테스트 포인트에서 성능 비교
# Compare: Model only vs Model + GP correction
# ============================================================

rng_test = np.random.default_rng(99)
n_test   = 80
alpha_test = rng_test.uniform(0, 25, n_test)
mach_test  = rng_test.uniform(0.5, 2.5, n_test)

cd_true_test  = cd_true(alpha_test, mach_test)
cd_model_test = cd_model(alpha_test, mach_test)

X_test_gp = np.column_stack([alpha_test / alpha_scale, mach_test / mach_scale])
delta_gp, delta_sigma = gp_aero.predict(X_test_gp)
cd_corrected_test = cd_model_test + delta_gp

rmse_model     = np.sqrt(np.mean((cd_model_test     - cd_true_test)**2))
rmse_corrected = np.sqrt(np.mean((cd_corrected_test - cd_true_test)**2))

print('=' * 45)
print(f'Model only  RMSE : {rmse_model:.5f}')
print(f'GP corrected RMSE: {rmse_corrected:.5f}')
print(f'개선율           : {(1-rmse_corrected/rmse_model)*100:.1f}%')
print('=' * 45)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
cd_lo = min(cd_true_test.min(), cd_model_test.min()) - 0.005
cd_hi = max(cd_true_test.max(), cd_model_test.max()) + 0.005
cd_range = [cd_lo, cd_hi]

for ax, pred, label, color in zip(
        axes,
        [cd_model_test, cd_corrected_test],
        ['선형 모델 (보정 전)', 'GP 보정 모델'],
        ['tomato', 'steelblue']):
    ax.scatter(cd_true_test, pred, c=color, alpha=0.65, s=45, edgecolors='none')
    ax.plot(cd_range, cd_range, 'k--', lw=1.5, label='Perfect prediction')
    ax.set_xlim(cd_range); ax.set_ylim(cd_range)
    ax.set_xlabel('True CD'); ax.set_ylabel('Predicted CD')
    ax.set_aspect('equal')
    rmse = np.sqrt(np.mean((pred - cd_true_test)**2))
    ax.set_title(f'{label}\nRMSE = {rmse:.5f}', pad=10)
    ax.legend()

plt.suptitle('예측 성능 비교: 선형 모델 vs GP 보정 모델 (테스트 80개)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

high_aoa   = alpha_test > 15
transonic  = (mach_test > 0.85) & (mach_test < 1.2)
rmse_m_ha  = np.sqrt(np.mean((cd_model_test[high_aoa]     - cd_true_test[high_aoa])**2))
rmse_c_ha  = np.sqrt(np.mean((cd_corrected_test[high_aoa] - cd_true_test[high_aoa])**2))
rmse_m_ts  = np.sqrt(np.mean((cd_model_test[transonic]     - cd_true_test[transonic])**2))
rmse_c_ts  = np.sqrt(np.mean((cd_corrected_test[transonic] - cd_true_test[transonic])**2))

print('\n--- 구간별 RMSE ---')
print(f'고받음각 (alpha>15 deg): Model={rmse_m_ha:.5f} -> GP={rmse_c_ha:.5f}  '
      f'({(1-rmse_c_ha/rmse_m_ha)*100:.0f}% 개선)')
print(f'천음속 (0.85<M<1.2)   : Model={rmse_m_ts:.5f} -> GP={rmse_c_ts:.5f}  '
      f'({(1-rmse_c_ts/rmse_m_ts)*100:.0f}% 개선)')

## 5. 유도 시뮬레이션에 적용

공력 모델 정확도가 **유도 성능(miss distance)**에 미치는 영향을 확인합니다.

*Evaluating the effect of aerodynamic model accuracy on guidance performance (miss distance).*

### 시뮬레이션 설정

- **비례항법(Proportional Navigation)** 유도: $a_c = N \\cdot V_c \\cdot \\dot{\\lambda}$
- 2D 수직면 교전 (Vertical plane engagement)
- 공력 오차가 실제 기동 응답에 영향 → miss distance 차이 발생

| 케이스 | 공력 모델 |
|--------|----------|
| Nominal | 선형 모델 (GP 없음) |
| True | 실제 비선형 공력 |
| GP-corrected | 선형 + GP 보정 |

In [ ]:
# ============================================================
# 간략화된 2D PN 유도 시뮬레이션
# 공력 모델 차이가 miss distance에 미치는 영향
# ============================================================

def run_engagement(aero_mode='nominal', gp=None,
                   a_scale=25.0, m_scale=2.5):
    """
    Simplified 2D point-mass PN engagement.
    aero_mode: 'nominal' | 'true' | 'gp_corrected'
    """
    g, rho, Sref, mass = 9.81, 0.9, 0.025, 50.0
    N_pn, a_sound = 4.0, 340.0
    x0, y0   = 0.0, 0.0
    vx0, vy0 = 480.0, 80.0
    xt, yt   = 3000.0, 1100.0
    dt, tmax = 0.002, 10.0
    N = int(tmax / dt)

    state = np.array([x0, y0, vx0, vy0])
    history = [state.copy()]

    def get_cd(alpha_deg, mach_val, mode):
        if mode == 'nominal':
            return cd_model(alpha_deg, mach_val)
        elif mode == 'true':
            return cd_true(alpha_deg, mach_val)
        else:  # gp_corrected
            cd_m = cd_model(alpha_deg, mach_val)
            x_in = np.array([[alpha_deg / a_scale, mach_val / m_scale]])
            delta, _ = gp.predict(x_in)
            return cd_m + float(delta[0])

    miss, t_impact = None, tmax

    for i in range(N):
        x, y, vx, vy = state
        rx, ry = xt - x, yt - y
        R = np.sqrt(rx**2 + ry**2)
        if R < 5.0:
            miss, t_impact = R, i * dt
            break

        V = np.sqrt(vx**2 + vy**2) + 1e-9
        lam_dot = (vx * ry - vy * rx) / R**2
        Vc = -(vx * rx + vy * ry) / R
        a_cmd = np.clip(N_pn * Vc * lam_dot, -40*g, 40*g)

        q_dyn    = 0.5 * rho * V**2
        cn_alpha = 10.0
        alpha_cmd_rad = np.clip(
            a_cmd * mass / (q_dyn * Sref * cn_alpha + 1e-9),
            -np.deg2rad(25), np.deg2rad(25))
        alpha_deg = abs(np.rad2deg(alpha_cmd_rad))
        mach_now  = V / a_sound

        CD      = get_cd(alpha_deg, mach_now, aero_mode)
        F_drag  = 0.5 * rho * V**2 * Sref * CD
        ax_drag = -F_drag / mass * (vx / V)
        ay_drag = -F_drag / mass * (vy / V) - g
        ax_lat  = -a_cmd * (vy / V)
        ay_lat  =  a_cmd * (vx / V)

        state = state + dt * np.array([vx, vy,
                                        ax_drag + ax_lat,
                                        ay_drag + ay_lat])
        history.append(state.copy())

    if miss is None:
        miss = np.sqrt((state[0]-xt)**2 + (state[1]-yt)**2)
    return np.array(history), miss, t_impact


print('시뮬레이션 실행 중...')
hist_nom,  miss_nom,  t_nom  = run_engagement('nominal')
hist_true, miss_true, t_true = run_engagement('true')
hist_gp,   miss_gp,   t_gp   = run_engagement(
    'gp_corrected', gp=gp_aero, a_scale=alpha_scale, m_scale=mach_scale)

print('\n' + '='*50)
print(f'{"케이스":<20}  {"Miss Distance":>14}  {"비행시간":>10}')
print('-'*50)
print(f'{"선형 모델 (명목)":<20}  {miss_nom:>12.2f} m  {t_nom:>8.3f} s')
print(f'{"True 비선형 공력":<20}  {miss_true:>12.2f} m  {t_true:>8.3f} s')
print(f'{"GP 보정 모델":<20}  {miss_gp:>12.2f} m  {t_gp:>8.3f} s')
print('='*50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for hist, lbl, col in [(hist_nom,  '선형 모델', 'tomato'),
                        (hist_true, 'True 공력', 'k'),
                        (hist_gp,   'GP 보정',   'steelblue')]:
    ax.plot(hist[:, 0], hist[:, 1], color=col, lw=1.8, label=lbl)
ax.scatter([3000], [1100], marker='*', c='gold', s=300,
           edgecolors='k', zorder=5, label='표적')
ax.set_xlabel('수평 거리 (m)')
ax.set_ylabel('고도 (m)')
ax.set_title('비행 궤적 비교', pad=10)
ax.legend()

ax2 = axes[1]
cases  = ['선형 모델\n(명목)', 'True 공력', 'GP 보정']
misses = [miss_nom, miss_true, miss_gp]
colors = ['tomato', 'gray', 'steelblue']
bars = ax2.bar(cases, misses, color=colors, edgecolor='k', linewidth=0.8, width=0.5)
for bar, val in zip(bars, misses):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f} m', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_ylabel('Miss Distance (m)')
ax2.set_title('Miss Distance 비교\n(낮을수록 좋음)', pad=10)
ax2.set_ylim(0, max(misses) * 1.3)

plt.suptitle('공력 모델 정확도가 유도 성능에 미치는 영향', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'\nGP 보정으로 miss distance {miss_nom - miss_gp:+.2f} m 변화')
print('(공력 모델의 정확도가 종말 유도 정밀도에 직접 영향)')

## 6. 한계와 고찰 (Limitations and Discussion)

### 실용화에서의 주요 장벽

**1. 학습 데이터 확보 문제**

현실에서 비행시험 횟수는 극히 제한적입니다. 유도탄 1발 = 수억 원.  
GP는 소수 데이터에서도 잘 동작하지만, **커버리지**가 보장되지 않음.

```
시뮬레이션: 30개 포인트  <- 이 연구
실제 CFT:   ~5-15회 비행  <- 현실적 한계
비행시험:   수 회          <- 최상위 데이터
```

**2. Extrapolation 위험**

GP는 학습 데이터 범위 **내에서만** 신뢰할 수 있습니다.  
불확실성 $\\sigma$가 커지면 보정값을 그대로 쓰면 안 됨 → **신뢰 임계값** 설정 필요.

$$\\text{Corrected CD} = \\begin{cases}
C_D^{\\text{model}} + \\hat{\\Delta C}_D & \\text{if } \\sigma < \\sigma_{\\text{threshold}} \\\\
C_D^{\\text{model}} & \\text{otherwise (fallback)}
\\end{cases}$$

**3. 계산 비용: O(N^3)**

GP의 핵심 병목은 행렬 역산:
- N = 100: ~0.001 sec → 실시간 가능
- N = 1000: ~1 sec → 어려움
- N = 10000: ~1000 sec → 불가

실시간 유도 적용을 위해서는 **Sparse GP** 또는 **GP lookup table** 사전 계산 필요.

**4. 커널 선택 sensitivity**

RBF vs Matérn vs Rational Quadratic — 물리적 prior 지식이 커널 선택에 중요.  
공력은 불연속성(충격파)이 있으므로 Matérn이 RBF보다 적합할 수 있음.

---

### 발전 방향

| 방향 | 방법 |
|------|------|
| 계산 효율 | Sparse GP (inducing points), Variational GP |
| 고차원 입력 | Neural Network kernel (Deep GP) |
| 물리 결합 | Physics-Informed GP (PDE constraints) |
| 온라인 학습 | Sequential/Recursive GP (비행 중 실시간 보정) |

*Physics-Informed GP — embedding PDE constraints (e.g., Navier-Stokes symmetry) as kernel priors — seems especially promising for aerodynamic applications.*

## 7. 정리 (Summary)

### 이번 연구에서 확인한 것

1. **GP는 공력 모델 보정에 적합한 도구**입니다.
   - 데이터가 적어도 동작 (30개로 의미있는 보정 가능)
   - 불확실성 정량화 → 공학적으로 직접 활용 가능

2. **Uncertainty quantification의 공학적 가치**
   - 단순 예측값보다 **예측 + 불확실성 구간**이 훨씬 유용
   - '이 영역은 데이터 없어서 모름' → 보수적 설계, 안전 마진 설정

3. **HILS 전 모델 보정 효과**
   - 공력 모델 정확도 향상 → 시뮬레이션 정확도 향상 → HILS 통과율 향상
   - GP 보정을 pre-HILS 단계에서 적용하면 시뮬-실물 gap 감소

4. **한계 인식**
   - 비행시험 데이터 희소성, 계산 비용, 외삽 위험 → 실용화 전 추가 연구 필요

---

### 개인적 소감

> 처음에는 '공력에 머신러닝?' 의구심이 있었는데,  
> GP의 *uncertainty-aware* 특성이 공학 응용에서 특히 강점이 있다는 걸 이해하게 됐습니다.  
> Kalman filter처럼, GP도 결국 '나의 모델이 얼마나 불확실한지'를 추적하는 도구입니다.  
> 그 관점에서 보면 기존 유도제어 접근과 자연스럽게 연결됩니다.
>
> *Initially skeptical of ML for aerodynamics, but GP's uncertainty-aware nature
> turned out to be its key strength. Like a Kalman filter, GP tracks 'how uncertain
> is my model' — and from that perspective, it connects naturally to GNC thinking.*

---

**참고 문헌 (References)**

- Rasmussen & Williams, *Gaussian Processes for Machine Learning*, MIT Press, 2006
- Zarchan, *Tactical and Strategic Missile Guidance*, 6th ed., AIAA, 2012
- Ko & Fox, 'GP-BayesFilters: Bayesian Filtering using Gaussian Process Prediction and Observation Models,' *IJRR*, 2009
- Gaussian Processes in Aerospace Applications: survey, *AIAA SciTech 2021*